# Sistema de Triaje Médico en Urgencias con Lógica Difusa
**Entrega 1 – Prototipo funcional** · Rama de IA: *Lógica difusa (sistema de inferencia Mamdani)*

> ⚠️ **Aviso:** prototipo académico con datos simulados. Los rangos, etiquetas y reglas son ilustrativos y **no** sustituyen el criterio de personal médico ni deben usarse con pacientes reales.

**Cómo ejecutarlo:** menú *Entorno de ejecución → Ejecutar todo*. Todo corre de principio a fin sin archivos adicionales (si no existe `datos_prueba.csv`, el notebook lo crea).

## 0. Marco teórico: ¿por qué la decisión es auditable y no una "caja negra"?

**El problema.** Una regla binaria clásica ("si temp ≥ 38.0 °C → urgencias; si 37.9 °C → sala de espera") trata a dos pacientes casi idénticos de forma opuesta por una décima de grado. La **lógica difusa** permite que un valor pertenezca *parcialmente* a varios conceptos a la vez (p. ej. 37.7 °C es "algo normal" y "algo fiebre moderada"), y así la prioridad cambia de forma **gradual**.

**Cómo decide el sistema (5 pasos, todos visibles en este notebook):**
1. **Fuzzificación:** convierte cada medición en grados de pertenencia (0 a 1) a etiquetas como `fiebre_moderada` o `dolor severo`.
2. **Evaluación de reglas:** cada regla `SI … Y … ENTONCES …` se activa con una fuerza igual al *mínimo* de sus condiciones.
3. **Implicación:** se recorta la etiqueta de salida a esa fuerza.
4. **Agregación:** se unen (máximo) las salidas de todas las reglas activas.
5. **Defuzzificación:** el centroide del área resultante da un número de prioridad de 0 a 100.

**Por qué es auditable (transparente):**
- **El conocimiento está en una lista de reglas legible** (sección 3), escrita en lenguaje cercano al clínico. Un médico puede leerla, discutirla y corregirla sin saber programar.
- **Cada resultado es trazable:** `triar_paciente()` imprime los grados de pertenencia y *qué reglas se activaron y con qué fuerza*. Se puede responder exactamente "¿por qué prioridad 63?".
- **Es determinista y reproducible:** mismos datos → mismo resultado, sin entrenamiento oculto ni pesos opacos como en una red neuronal.
- **Es verificable:** las gráficas de pertenencia permiten revisar visualmente si cada concepto ("fiebre alta", "dolor severo") está definido de forma razonable.

In [ ]:
# En Google Colab: instalar scikit-fuzzy (ejecutar una sola vez)
!pip install scikit-fuzzy

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import skfuzzy as fuzz   # scikit-fuzzy: funciones de pertenencia, interpolación y defuzzificación

plt.rcParams["figure.dpi"] = 110

## 1. Universos del discurso y funciones de pertenencia
Variables de entrada: `temperatura` (35–41 °C), `presion_arterial` sistólica (70–190 mmHg), `nivel_dolor` (0–10). Variable de salida: `prioridad_atencion` (0–100).

In [ ]:
# ---------------------------------------------------------------
# 1) UNIVERSOS DEL DISCURSO Y FUNCIONES DE PERTENENCIA
# ---------------------------------------------------------------
# Cada variable tiene un universo (rango de valores posibles) y etiquetas
# lingüísticas. Cada etiqueta se define con una función de pertenencia:
#   ("tri",  [a, b, c])     -> triangular
#   ("trap", [a, b, c, d])  -> trapezoidal (a==b o c==d forman "hombros";
#   en los extremos se deja c=d fuera del universo para que el hombro valga 1 hasta el borde)

VARIABLES = {
    "temperatura": {
        "unidad": "°C",
        "universo": np.round(np.arange(35.0, 41.01, 0.1), 1),
        "etiquetas": {
            "baja":           ("trap", [35.0, 35.0, 35.5, 36.5]),
            "normal":         ("trap", [36.0, 36.5, 37.2, 37.8]),
            "fiebre_moderada":("tri",  [37.4, 38.2, 39.0]),
            "fiebre_alta":    ("trap", [38.6, 39.5, 42.0, 42.0]),
        },
    },
    "presion_arterial": {
        "unidad": "mmHg (sistólica)",
        "universo": np.arange(70, 190.1, 1.0),
        "etiquetas": {
            "baja":     ("trap", [70, 70, 85, 100]),
            "normal":   ("trap", [90, 105, 120, 135]),
            "elevada":  ("tri",  [125, 140, 160]),
            "muy_alta": ("trap", [150, 170, 200, 200]),
        },
    },
    "nivel_dolor": {
        "unidad": "escala 0-10",
        "universo": np.round(np.arange(0, 10.01, 0.1), 1),
        "etiquetas": {
            "leve":     ("trap", [0, 0, 2, 4]),
            "moderado": ("tri",  [3, 5, 7]),
            "severo":   ("trap", [6, 8, 11, 11]),
        },
    },
    "prioridad_atencion": {
        "unidad": "0-100",
        "universo": np.arange(0, 100.01, 0.5),
        "etiquetas": {
            "baja":       ("trap", [0, 0, 15, 30]),
            "media_baja": ("tri",  [20, 35, 50]),
            "media_alta": ("tri",  [45, 65, 80]),
            "critica":    ("trap", [70, 85, 101, 101]),
        },
    },
}

def construir_mf(universo, tipo, params):
    """Evalúa una función de pertenencia sobre el universo."""
    return fuzz.trimf(universo, params) if tipo == "tri" else fuzz.trapmf(universo, params)

# Se precalcula la curva de cada etiqueta: MF[variable][etiqueta] -> arreglo
MF = {
    var: {etq: construir_mf(d["universo"], *definicion)
          for etq, definicion in d["etiquetas"].items()}
    for var, d in VARIABLES.items()
}
print("Variables definidas:", {v: list(d["etiquetas"]) for v, d in VARIABLES.items()})

## 2. Visualización de las funciones de pertenencia
Observa cómo las etiquetas **se traslapan**: en esas zonas de traslape está la "zona gris" que la lógica binaria no puede representar.

In [ ]:
# ---------------------------------------------------------------
# 2) VISUALIZACIÓN DE LAS FUNCIONES DE PERTENENCIA
# ---------------------------------------------------------------
def graficar_membresias(valores=None):
    """Grafica las funciones de pertenencia de las 4 variables.
    Si se pasan `valores` (dict con las 3 entradas), marca el caso del paciente
    con una línea vertical para ver en qué 'zona gris' cae."""
    fig, axes = plt.subplots(2, 2, figsize=(12, 7))
    for ax, (var, d) in zip(axes.ravel(), VARIABLES.items()):
        for etq, curva in MF[var].items():
            ax.plot(d["universo"], curva, linewidth=2, label=etq)
        if valores and var in valores:
            ax.axvline(valores[var], color="k", linestyle="--", linewidth=1.2)
            ax.text(valores[var], 1.03, f"{valores[var]}", ha="center", fontsize=9)
        ax.set_title(f"{var} [{d['unidad']}]")
        ax.set_ylabel("Grado de pertenencia")
        ax.set_ylim(-0.02, 1.4)
        ax.legend(loc="upper center", ncol=2, fontsize=8)
        ax.grid(alpha=0.3)
    fig.suptitle("Funciones de pertenencia del sistema de triaje difuso", fontsize=13)
    plt.tight_layout()
    plt.show()

graficar_membresias()

## 3. Base de reglas
23 reglas `SI … Y … ENTONCES …` que simulan el criterio médico (casos críticos, media-alta, media-baja y baja). Se verificó que **toda combinación posible** de entradas activa al menos una regla (cobertura completa).

In [ ]:
# ---------------------------------------------------------------
# 3) BASE DE REGLAS  (SI ... Y ... ENTONCES ...)
# ---------------------------------------------------------------
# Esta lista ES el conocimiento del sistema. Cualquier médico puede leerla,
# criticarla y modificarla: ahí está la transparencia (no es una "caja negra").
# Todas las reglas usan el operador Y (mínimo). El operador O se expresa
# escribiendo varias reglas con la misma consecuencia.

REGLAS = [
    # --- Casos críticos ---
    ("R1",  [("temperatura", "fiebre_alta"),    ("nivel_dolor", "severo")],        "critica"),
    ("R2",  [("presion_arterial", "muy_alta"),  ("nivel_dolor", "severo")],        "critica"),
    ("R3",  [("presion_arterial", "baja"),      ("temperatura", "fiebre_alta")],   "critica"),
    ("R4",  [("presion_arterial", "baja"),      ("nivel_dolor", "severo")],        "critica"),
    ("R5",  [("temperatura", "baja"),           ("presion_arterial", "baja")],     "critica"),
    # --- Prioridad media-alta ---
    ("R6",  [("temperatura", "fiebre_moderada"),("presion_arterial", "elevada")],  "media_alta"),
    ("R7",  [("temperatura", "fiebre_moderada"),("nivel_dolor", "moderado")],      "media_alta"),
    ("R8",  [("temperatura", "fiebre_moderada"),("nivel_dolor", "severo")],        "media_alta"),
    ("R9",  [("temperatura", "fiebre_alta"),    ("nivel_dolor", "moderado")],      "media_alta"),
    ("R10", [("temperatura", "fiebre_alta"),    ("nivel_dolor", "leve")],          "media_alta"),
    ("R11", [("presion_arterial", "elevada"),   ("nivel_dolor", "moderado")],      "media_alta"),
    ("R12", [("presion_arterial", "muy_alta"),  ("nivel_dolor", "leve")],          "media_alta"),
    ("R13", [("presion_arterial", "muy_alta"),  ("nivel_dolor", "moderado")],      "media_alta"),
    ("R14", [("presion_arterial", "baja"),      ("nivel_dolor", "moderado")],      "media_alta"),
    ("R15", [("temperatura", "normal"),         ("nivel_dolor", "severo")],        "media_alta"),
    ("R15b",[("temperatura", "baja"),           ("nivel_dolor", "severo")],        "media_alta"),
    # --- Prioridad media-baja ---
    ("R16", [("temperatura", "fiebre_moderada"),("nivel_dolor", "leve")],          "media_baja"),
    ("R17", [("temperatura", "normal"),         ("nivel_dolor", "moderado")],      "media_baja"),
    ("R18", [("temperatura", "baja"),           ("nivel_dolor", "leve")],          "media_baja"),
    ("R19", [("temperatura", "baja"),           ("nivel_dolor", "moderado")],      "media_baja"),
    ("R20", [("presion_arterial", "elevada"),   ("nivel_dolor", "leve")],          "media_baja"),
    ("R21", [("presion_arterial", "baja"),      ("nivel_dolor", "leve")],          "media_baja"),
    # --- Prioridad baja ---
    ("R22", [("temperatura", "normal"), ("presion_arterial", "normal"), ("nivel_dolor", "leve")], "baja"),
]

def texto_regla(regla):
    """Devuelve la regla en lenguaje natural."""
    nombre, antecedentes, consecuente = regla
    cond = " Y ".join(f"{v} es {e}" for v, e in antecedentes)
    return f"{nombre}: SI {cond} ENTONCES prioridad_atencion es {consecuente}"

for r in REGLAS:
    print(texto_regla(r))

## 4. Motor de inferencia
Implementación explícita de los 5 pasos de Mamdani usando `skfuzzy` (funciones de pertenencia, interpolación y defuzzificación por centroide).

In [ ]:
# ---------------------------------------------------------------
# 4) MOTOR DE INFERENCIA (tipo Mamdani, paso a paso)
# ---------------------------------------------------------------
# Paso 1. Fuzzificación: número exacto -> grados de pertenencia (0 a 1)
# Paso 2. Evaluación de reglas: operador Y = mínimo de los grados
# Paso 3. Implicación: se "recorta" la etiqueta de salida a la fuerza de la regla
# Paso 4. Agregación: se unen todas las salidas recortadas (máximo)
# Paso 5. Defuzzificación: centroide del área -> número exacto de prioridad

def fuzzificar(entradas):
    """Paso 1: grado de pertenencia de cada valor a cada etiqueta."""
    grados = {}
    for var, valor in entradas.items():
        d = VARIABLES[var]
        grados[var] = {etq: float(fuzz.interp_membership(d["universo"], curva, valor))
                       for etq, curva in MF[var].items()}
    return grados

def inferir(entradas):
    """Ejecuta el sistema completo. Devuelve el resultado y la explicación."""
    # Validación de rangos (evita datos fuera del universo del discurso)
    for var, valor in entradas.items():
        u = VARIABLES[var]["universo"]
        if not (u[0] <= valor <= u[-1]):
            raise ValueError(f"{var}={valor} fuera de rango [{u[0]:.1f}, {u[-1]:.1f}]")

    grados = fuzzificar(entradas)                                   # Paso 1
    universo_y = VARIABLES["prioridad_atencion"]["universo"]
    agregada = np.zeros_like(universo_y)
    disparos = []
    for nombre, antecedentes, consecuente in REGLAS:
        fuerza = min(grados[v][e] for v, e in antecedentes)         # Paso 2 (Y = min)
        recorte = np.fmin(fuerza, MF["prioridad_atencion"][consecuente])   # Paso 3
        agregada = np.fmax(agregada, recorte)                       # Paso 4 (max)
        if fuerza > 0:
            disparos.append((nombre, fuerza, consecuente))
    if agregada.sum() == 0:
        raise RuntimeError("Ninguna regla se activó: revisar la cobertura de la base de reglas.")
    prioridad = float(fuzz.defuzz(universo_y, agregada, "centroid"))  # Paso 5
    disparos.sort(key=lambda t: t[1], reverse=True)
    return {"entradas": entradas, "grados": grados, "disparos": disparos,
            "agregada": agregada, "prioridad": prioridad}

## 5. Interpretación clínica y función de prueba
`triar_paciente(temperatura, presion_arterial, nivel_dolor)` recibe un caso hipotético, muestra el valor numérico exacto, la interpretación clínica, las reglas activadas y las gráficas.

In [ ]:
# ---------------------------------------------------------------
# 5) INTERPRETACIÓN CLÍNICA Y FUNCIÓN DE PRUEBA
# ---------------------------------------------------------------
def interpretar(prioridad):
    """Traduce el valor numérico (0-100) a una recomendación de atención."""
    if prioridad < 25:
        return "BAJA - Sala de espera general; reevaluar periódicamente."
    elif prioridad < 50:
        return "MEDIA-BAJA - Atención programada; vigilar signos vitales."
    elif prioridad < 75:
        return "MEDIA-ALTA - Atención pronta; pasar a valoración médica."
    return "CRÍTICA - Atención inmediata en urgencias."

def graficar_resultado(res):
    """Muestra el área agregada y el valor defuzzificado (centroide)."""
    y = VARIABLES["prioridad_atencion"]["universo"]
    fig, ax = plt.subplots(figsize=(9, 3.8))
    for etq, curva in MF["prioridad_atencion"].items():
        ax.plot(y, curva, "--", linewidth=1, alpha=0.6, label=etq)
    ax.fill_between(y, res["agregada"], color="tab:orange", alpha=0.45, label="salida agregada")
    ax.axvline(res["prioridad"], color="k", linewidth=2)
    ax.text(res["prioridad"] + 1, 0.9, f"{res['prioridad']:.1f}", fontsize=11, fontweight="bold")
    ax.set_title("Resultado defuzzificado (centroide)")
    ax.set_xlabel("prioridad_atencion (0-100)"); ax.set_ylabel("Pertenencia")
    ax.set_ylim(0, 1.05); ax.legend(fontsize=8, loc="upper left"); ax.grid(alpha=0.3)
    plt.tight_layout(); plt.show()

def triar_paciente(temperatura, presion_arterial, nivel_dolor, graficar=True, explicar=True):
    """Evalúa un paciente hipotético y muestra el resultado en consola."""
    entradas = {"temperatura": temperatura,
                "presion_arterial": presion_arterial,
                "nivel_dolor": nivel_dolor}
    res = inferir(entradas)
    print("=" * 62)
    print(f"PACIENTE: temp={temperatura} °C | presión={presion_arterial} mmHg | dolor={nivel_dolor}/10")
    print("=" * 62)
    if explicar:
        print("Grados de pertenencia (fuzzificación):")
        for var, etqs in res["grados"].items():
            activos = {e: round(g, 2) for e, g in etqs.items() if g > 0}
            print(f"  {var:17s} -> {activos}")
        print("\nReglas activadas (de mayor a menor fuerza):")
        for nombre, fuerza, cons in res["disparos"]:
            regla = next(r for r in REGLAS if r[0] == nombre)
            print(f"  [{fuerza:.2f}] {texto_regla(regla)}")
    print(f"\n>>> PRIORIDAD DE ATENCIÓN: {res['prioridad']:.2f} / 100")
    print(f">>> INTERPRETACIÓN CLÍNICA: {interpretar(res['prioridad'])}\n")
    if graficar:
        graficar_membresias(entradas)
        graficar_resultado(res)
    return res

# Caso del enunciado
resultado = triar_paciente(temperatura=38.2, presion_arterial=135, nivel_dolor=6)

## 6. La "zona gris": sin saltos bruscos
El sistema no cambia de golpe en 38.0 °C; la prioridad sube gradualmente mientras la temperatura deja de ser "normal" y se vuelve "fiebre moderada".

In [ ]:
# ---------------------------------------------------------------
# 6) DEMOSTRACIÓN DE LA "ZONA GRIS"
# ---------------------------------------------------------------
# Regla binaria clásica: 37.9 °C -> sala de espera, 38.0 °C -> urgencias (salto brusco de 0.1 °C).
# Con lógica difusa NO hay salto: la prioridad cambia de forma gradual conforme la
# temperatura deja de ser "normal" y empieza a ser "fiebre_moderada".
print("Prioridad al subir la temperatura (presión=120 mmHg, dolor=5):")
for t in (37.2, 37.4, 37.6, 37.8, 37.9, 38.0, 38.1, 38.2):
    p = inferir({"temperatura": t, "presion_arterial": 120, "nivel_dolor": 5})["prioridad"]
    print(f"  {t:.1f} °C -> prioridad {p:5.1f}  ({interpretar(p).split(' - ')[0]})")

temps = np.round(np.arange(35.0, 41.01, 0.1), 1)
fig, ax = plt.subplots(figsize=(9, 4))
for dolor in (2, 5, 8):
    curva = [inferir({"temperatura": float(t), "presion_arterial": 120, "nivel_dolor": dolor})["prioridad"]
             for t in temps]
    ax.plot(temps, curva, linewidth=2, label=f"dolor = {dolor}")
ax.axvline(38.0, color="gray", linestyle=":", label="umbral binario clásico (38.0 °C)")
ax.set_xlabel("Temperatura (°C)"); ax.set_ylabel("Prioridad de atención (0-100)")
ax.set_title("Respuesta gradual del sistema difuso (presión = 120 mmHg)")
ax.legend(); ax.grid(alpha=0.3); plt.tight_layout(); plt.show()

## 7. Datos de prueba y evaluación por lotes
Pacientes **simulados** (`datos_prueba.csv`) evaluados con Pandas y ordenados como cola de triaje.

In [ ]:
# ---------------------------------------------------------------
# 7) DATOS DE PRUEBA Y EVALUACIÓN POR LOTES (Pandas)
# ---------------------------------------------------------------
# Datos SIMULADOS (no son pacientes reales). Si el archivo datos_prueba.csv no está
# junto al notebook, se crea automáticamente con estos casos.
import os
CASOS = [
    ("P01", 36.8, 118, 1), ("P02", 37.9, 120, 5), ("P03", 38.0, 120, 5),
    ("P04", 38.2, 135, 6), ("P05", 39.6, 110, 8), ("P06", 36.4, 88, 3),
    ("P07", 37.0, 165, 7), ("P08", 35.4, 105, 2),  ("P09", 38.8, 145, 5),
    ("P10", 37.2, 95, 9),  ("P11", 40.2, 80, 9),  ("P12", 36.9, 178, 8),
]
if not os.path.exists("datos_prueba.csv"):
    pd.DataFrame(CASOS, columns=["id", "temperatura", "presion_arterial", "nivel_dolor"]) \
      .to_csv("datos_prueba.csv", index=False)

df = pd.read_csv("datos_prueba.csv")
df["prioridad"] = df.apply(lambda f: inferir({"temperatura": f.temperatura,
                                              "presion_arterial": f.presion_arterial,
                                              "nivel_dolor": f.nivel_dolor})["prioridad"], axis=1).round(1)
df["nivel"] = df["prioridad"].apply(lambda p: interpretar(p).split(" - ")[0])
df = df.sort_values("prioridad", ascending=False).reset_index(drop=True)
try:
    display(df)                      # tabla bonita en Colab / Jupyter
except NameError:
    print(df.to_string(index=False))

colores = {"BAJA": "tab:green", "MEDIA-BAJA": "gold", "MEDIA-ALTA": "tab:orange", "CRÍTICA": "tab:red"}
plt.figure(figsize=(9, 4))
plt.barh(df["id"][::-1], df["prioridad"][::-1], color=[colores[n] for n in df["nivel"][::-1]])
plt.xlabel("Prioridad de atención (0-100)"); plt.title("Cola de triaje ordenada por prioridad")
plt.grid(axis="x", alpha=0.3); plt.tight_layout(); plt.show()

## 8. Prueba libre
Modifica los tres valores y ejecuta la celda.

In [ ]:
# ---------------------------------------------------------------
# 8) PRUEBA LIBRE: cambia los valores y vuelve a ejecutar
# ---------------------------------------------------------------
_ = triar_paciente(temperatura=39.1, presion_arterial=95, nivel_dolor=7)

## Créditos y licencias
- **scikit-fuzzy** (BSD-3-Clause), **NumPy** (BSD), **pandas** (BSD-3-Clause), **Matplotlib** (licencia tipo PSF), ejecutado en **Google Colab**.
- Código base generado con asistencia de IA (Claude) a partir de un prompt del equipo; ver `README.md` (declaración de uso de IA) y `bitacora_prompts.md`.
- Datos de prueba: simulados por el equipo, no provienen de pacientes reales.